In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Uninstall existing PyTorch and torchvision packages to prevent conflicts
!pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [ ]:
# Install the latest stable version of PyTorch and torchvision compatible with CUDA
# (adjust the version if you have specific requirements)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 76.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 54.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 127.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/

In [ ]:
import os
import re
import glob
from collections import defaultdict, Counter
import torch.quantization


import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import mobilenet_v3_small


def parse_patch_index(filename: str):
    """
    Extract patch number from filename.
    Accepts patterns like: _p1, -p2, _patch3, patch4
    Returns int or None.
    """
    name = os.path.splitext(os.path.basename(filename))[0].lower()

    m = re.search(r'(?:^|[_\- ])(?:p|patch)\s*(\d+)$', name)
    if m:
        return int(m.group(1))
    return None

def get_group_id(filename: str):
    """
    Remove the patch suffix to get "original image id"
    Example:
      MI(1)_p1.jpg -> MI(1)
      abc_patch4.png -> abc
    """
    base = os.path.splitext(os.path.basename(filename))[0]
    # remove suffix like _p1 / _patch2 / -p3 / patch4 at end
    base = re.sub(r'([_\- ]?(p|patch)\s*\d+)$', '', base, flags=re.IGNORECASE)
    return base



class PatchFusionFolder(Dataset):
    def __init__(self, root_dir, transform=None, patches_per_image=4, strict=True):
        """
        root_dir: Train/Val/Test folder with class subfolders (ImageFolder style)
        strict=True: keep only samples that have exactly patches_per_image
        """
        self.root_dir = root_dir
        self.transform = transform
        self.patches_per_image = patches_per_image
        self.strict = strict

        base = ImageFolder(root=root_dir)
        self.classes = base.classes
        self.class_to_idx = base.class_to_idx
        samples = base.samples  # list of (path, label)

        # group by (label, group_id)
        grouped = defaultdict(list)
        for path, label in samples:
            pid = parse_patch_index(path)
            gid = get_group_id(path)
            if pid is None:
                # skip files that don't look like patches
                continue
            grouped[(label, gid)].append((pid, path))

        self.items = []
        for (label, gid), lst in grouped.items():
            # sort patches by pid
            lst = sorted(lst, key=lambda x: x[0])  # [(1,path),(2,path)...]

            # build dict pid->path
            pid_to_path = {pid: p for pid, p in lst}

            # we expect patches 1..N
            needed = list(range(1, patches_per_image + 1))
            has_all = all(n in pid_to_path for n in needed)

            if strict:
                if has_all and len(pid_to_path) == patches_per_image:
                    paths = [pid_to_path[n] for n in needed]
                    self.items.append((paths, label))
            else:
                # allow extra patches, just take first N if available
                if has_all:
                    paths = [pid_to_path[n] for n in needed]
                    self.items.append((paths, label))

        if len(self.items) == 0:
            raise ValueError(
                "No grouped samples found.\n"
                "Check patch file naming: must end with _p1.._p4 or _patch1.._patch4, etc."
            )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        patch_paths, label = self.items[idx]

        patches = []
        for p in patch_paths:
            img = Image.open(p).convert("RGB")
            if self.transform:
                img = self.transform(img)
            patches.append(img)

        x = torch.stack(patches, dim=0)           # (4,3,224,224)
        y = torch.tensor(label, dtype=torch.long) # scalar
        return x, y

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
#Define data transformations
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
#train_dataset = datasets.ImageFolder(root="/content/drive/MyDrive/ProjectDataSet/dataset_split_per_class/Train" , transform=transform)
#test_dataset = datasets.ImageFolder(root="/content/drive/MyDrive/ProjectDataSet/dataset_split_per_class/Test", transform=transform)


train_dataset=PatchFusionFolder(root_dir="/content/drive/MyDrive/ProjectDataSet/dataset_split_per_class/Train" , transform=transform ,patches_per_image=4, strict=True)
val_dataset=PatchFusionFolder(root_dir="/content/drive/MyDrive/ProjectDataSet/dataset_split_per_class/Val" , transform=transform ,patches_per_image=4, strict=True)
test_dataset=PatchFusionFolder(root_dir="/content/drive/MyDrive/ProjectDataSet/dataset_split_per_class/Test" , transform=transform ,patches_per_image=4, strict=True)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader=DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
!pip uninstall -y transformers huggingface_hub
!pip install -U transformers huggingface_hub
from transformers import AutoImageProcessor, AutoModelForImageClassification

model=AutoModelForImageClassification.from_pretrained("microsoft/swin-tiny-patch4-window7-224" , num_labels=4 ,  ignore_mismatched_sizes=True)


Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: huggingface_hub 1.17.0
Uninstalling huggingface_hub-1.17.0:
  Successfully uninstalled huggingface_hub-1.17.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 52.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([4])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
batch = next(iter(train_loader))
patches, labels = batch

print(patches.shape)
print(labels.shape)

torch.Size([32, 4, 3, 224, 224])
torch.Size([32])


In [ ]:
import torch.optim as optim  #train cell
import torch.nn as nn
import torch

#The loss function
crit=nn.CrossEntropyLoss()
#Optimizer
optimizer=optim.Adam(model.parameters(),lr=1e-5)
print("training started")
epochs=10

for epoch in range(epochs):
  # Training Phase
  model.train()
  running_loss=0
  correct=0
  total=0
  for inputs, labels in train_loader:
    optimizer.zero_grad()

    B, P, C, H, W = inputs.shape
    inputs_merged = inputs.view(B * P, C, H, W)  # [B*4, 3, 224, 224]


    outputs_model=model(inputs_merged)
    logits=outputs_model.logits

    # Original loss calculation
    outputs = logits.view(B, P * logits.shape[1])
    loss=crit(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss+=loss.item()

    # Accuracy calculation (averaging patch logits as done in evaluation)
    with torch.no_grad():
        logits_acc = logits.view(B, P, -1).mean(dim=1)
        _, predicted = torch.max(logits_acc, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

  epoch_loss = running_loss / len(train_loader)
  epoch_acc = 100 * correct / total

  # Validation Phase
  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0
  with torch.no_grad():
      for inputs, labels in validation_loader:
          B, P, C, H, W = inputs.shape
          inputs_merged = inputs.view(B * P, C, H, W)

          outputs_model = model(inputs_merged)
          logits = outputs_model.logits

          # Validation loss
          outputs = logits.view(B, P * logits.shape[1])
          loss = crit(outputs, labels)
          val_loss += loss.item()

          # Validation accuracy
          logits_acc = logits.view(B, P, -1).mean(dim=1)
          _, predicted = torch.max(logits_acc, 1)
          val_total += labels.size(0)
          val_correct += (predicted == labels).sum().item()

  epoch_val_loss = val_loss / len(validation_loader)
  epoch_val_acc = 100 * val_correct / val_total

  print(f"Epoch[{epoch+1}/{epochs}], Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%")



training started
Epoch[1/10], Train Loss: 0.2132, Train Acc: 74.80%, Val Loss: 0.3106, Val Acc: 65.96%
Epoch[2/10], Train Loss: 0.1685, Train Acc: 78.17%, Val Loss: 0.2918, Val Acc: 72.34%
Epoch[3/10], Train Loss: 0.1339, Train Acc: 78.98%, Val Loss: 0.1905, Val Acc: 86.17%
Epoch[4/10], Train Loss: 0.1115, Train Acc: 84.50%, Val Loss: 0.2211, Val Acc: 87.23%
Epoch[5/10], Train Loss: 0.1078, Train Acc: 82.88%, Val Loss: 0.2022, Val Acc: 89.36%
Epoch[6/10], Train Loss: 0.1224, Train Acc: 80.05%, Val Loss: 0.2288, Val Acc: 81.91%
Epoch[7/10], Train Loss: 0.0837, Train Acc: 83.15%, Val Loss: 0.1198, Val Acc: 82.98%
Epoch[8/10], Train Loss: 0.0629, Train Acc: 90.43%, Val Loss: 0.1071, Val Acc: 93.62%
Epoch[9/10], Train Loss: 0.0524, Train Acc: 93.26%, Val Loss: 0.0913, Val Acc: 91.49%
Epoch[10/10], Train Loss: 0.0601, Train Acc: 94.61%, Val Loss: 0.0691, Val Acc: 93.62%


In [ ]:
import torch

def evaluate_accuracy(loader, eval_model):
    eval_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            B, P, C, H, W = inputs.shape
            inputs_merged = inputs.view(B * P, C, H, W)

            outputs = eval_model(inputs_merged)
            logits = outputs.logits
            logits = logits.view(B, P, -1)
            logits = logits.mean(dim=1)

            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

test_acc_original = evaluate_accuracy(test_loader, model)
print(f"Test Accuracy (Original Model): {test_acc_original:.2f}%")

#test_acc_quantized = evaluate_accuracy(test_loader, quantized_model)
#print(f"Test Accuracy (Quantized Model): {test_acc_quantized:.2f}%")

Test Accuracy (Original Model): 97.80%


In [ ]:
import torch

# Path to your saved weights
weights_path = '/content/drive/MyDrive/swin_ecg_97.pth'

# Load the state dictionary
try:
    # Use map_location='cpu' if you are not using a GPU right now to avoid errors
    checkpoint = torch.load(weights_path, map_location=torch.device('cpu'))

    # Sometimes weights are saved inside a dictionary under a 'model' or 'state_dict' key
    if 'state_dict' in checkpoint:
        saved_state_dict = checkpoint['state_dict']
    elif 'model' in checkpoint:
        saved_state_dict = checkpoint['model']
    else:
        saved_state_dict = checkpoint

    # Map the old HuggingFace keys to the new format to fix the mismatch
    mapped_state_dict = {}
    for key, value in saved_state_dict.items():
        new_key = key
        # Attention mapping
        new_key = new_key.replace("attention.self.query", "attention.q_proj")
        new_key = new_key.replace("attention.self.key", "attention.k_proj")
        new_key = new_key.replace("attention.self.value", "attention.v_proj")
        new_key = new_key.replace("attention.output.dense", "attention.o_proj")

        # Relative position mapping
        new_key = new_key.replace("attention.self.relative_position_bias_table", "attention.relative_position_bias.relative_position_bias_table")
        new_key = new_key.replace("attention.self.relative_position_index", "attention.relative_position_bias.relative_position_index")

        # MLP mapping
        new_key = new_key.replace("intermediate.dense", "mlp.fc1")
        new_key = new_key.replace("output.dense", "mlp.fc2")

        mapped_state_dict[new_key] = value

    # Load the mapped weights into your base model
    missing, unexpected = model.load_state_dict(mapped_state_dict, strict=False)
    print("Weights loaded successfully! Your model is ready to use.")

    if missing:
        print(f"\nNote: The following keys were missing (this is normal if it's just the classifier head): \n{missing}")

    # Set to evaluation mode if you plan to do inference/testing
    model.eval()
except Exception as e:
    print(f"Error loading weights: {e}")


In [ ]:
import torch

weights_path = '/content/drive/MyDrive/swin_ecg_97.pth'
checkpoint = torch.load(weights_path, map_location='cpu')

# Extract state_dict if it's nested
if 'state_dict' in checkpoint:
    saved_state_dict = checkpoint['state_dict']
elif 'model' in checkpoint:
    saved_state_dict = checkpoint['model']
else:
    saved_state_dict = checkpoint

saved_keys = list(saved_state_dict.keys())
model_keys = list(model.state_dict().keys())

print("=== First 10 keys in SAVED weights ===")
for k in saved_keys[:10]:
    print(k)

print("\n=== First 10 keys in CURRENT model ===")
for k in model_keys[:10]:
    print(k)

print(f"\nTotal keys in saved model: {len(saved_keys)}")
print(f"Total keys in current model: {len(model_keys)}")


In [ ]:
print("Classes and their labels:")
for class_name, label in train_dataset.class_to_idx.items():
    print(f"Class: {class_name}, Label: {label}")

In [ ]:
import torch
import torch.nn as nn
import os
import time

# Ensure model is in eval mode on CPU
model.to('cpu')
model.eval()

# --- Evaluate BEFORE PTQ ---
correct_before = 0
total_before = 0

start_time_before = time.time()
with torch.no_grad():
    for inputs, labels in test_loader:
        B, P, C, H, W = inputs.shape
        inputs_merged = inputs.view(B * P, C, H, W)

        outputs = model(inputs_merged)
        logits = outputs.logits
        logits = logits.view(B, P, -1)
        logits = logits.mean(dim=1)

        _, predicted = torch.max(logits, 1)

        total_before += labels.size(0)
        correct_before += (predicted == labels).sum().item()
end_time_before = time.time()
time_before = end_time_before - start_time_before

accuracy_before = 100 * correct_before / total_before
print(f"Test Accuracy BEFORE Quantization: {accuracy_before:.2f}%")
print(f"Inference Time BEFORE Quantization: {time_before:.2f} seconds")

# --- Dynamic Quantization ---
print("\nApplying Dynamic Quantization...")
# Dynamically quantize only the nn.Linear layers (which make up the bulk of the transformer)
quantized_model = torch.ao.quantization.quantize_dynamic(
    model,
    {nn.Linear},
    dtype=torch.qint8
)

# --- Evaluate AFTER Dynamic Quantization ---
correct_after = 0
total_after = 0

start_time_after = time.time()
with torch.no_grad():
    for inputs, labels in test_loader:
        B, P, C, H, W = inputs.shape
        inputs_merged = inputs.view(B * P, C, H, W)

        outputs = quantized_model(inputs_merged)
        logits = outputs.logits
        logits = logits.view(B, P, -1)
        logits = logits.mean(dim=1)

        _, predicted = torch.max(logits, 1)

        total_after += labels.size(0)
        correct_after += (predicted == labels).sum().item()
end_time_after = time.time()
time_after = end_time_after - start_time_after

accuracy_after = 100 * correct_after / total_after
print(f"Test Accuracy AFTER Dynamic Quantization: {accuracy_after:.2f}%")
print(f"Inference Time AFTER Dynamic Quantization: {time_after:.2f} seconds")

# --- Check Size & Speed Reduction ---
def print_size_of_model(m):
    torch.save(m.state_dict(), "temp.p")
    size = os.path.getsize("temp.p") / 1e6
    os.remove("temp.p")
    return size

print(f"\nModel Size BEFORE: {print_size_of_model(model):.2f} MB")
print(f"Model Size AFTER: {print_size_of_model(quantized_model):.2f} MB")
print(f"Speedup Factor: {time_before / time_after:.2f}x")

In [ ]:
import torch

def evaluate_accuracy(loader, eval_model):
    eval_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            B, P, C, H, W = inputs.shape
            inputs_merged = inputs.view(B * P, C, H, W)

            outputs = eval_model(inputs_merged)
            logits = outputs.logits
            logits = logits.view(B, P, -1)
            logits = logits.mean(dim=1)

            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# Evaluate test accuracy BEFORE quantization
test_acc_before = evaluate_accuracy(test_loader, model)
print(f"Test Accuracy BEFORE Quantization: {test_acc_before:.2f}%")

# Evaluate test accuracy AFTER quantization
test_acc_after = evaluate_accuracy(test_loader, quantized_model)
print(f"Test Accuracy AFTER Quantization: {test_acc_after:.2f}%")


In [ ]:
def evaluate_accuracy(loader, eval_model):
    eval_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            B, P, C, H, W = inputs.shape
            inputs_merged = inputs.view(B * P, C, H, W)

            outputs = eval_model(inputs_merged)
            logits = outputs.logits
            logits = logits.view(B, P, -1)
            logits = logits.mean(dim=1)

            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

print("Evaluating accuracies BEFORE quantization ")
train_acc_before = evaluate_accuracy(train_loader, model)
print(f"Training Accuracy (Before): {train_acc_before:.2f}%")
val_acc_before = evaluate_accuracy(validation_loader, model)
print(f"Validation Accuracy (Before): {val_acc_before:.2f}%")

print("\nEvaluating accuracies AFTER dynamic quantization ")
# Make sure the quantized_model variable is in memory (from running the previous cell)
try:
    train_acc_after = evaluate_accuracy(train_loader, quantized_model)
    print(f"Training Accuracy (After): {train_acc_after:.2f}%")
    val_acc_after = evaluate_accuracy(validation_loader, quantized_model)
    print(f"Validation Accuracy (After): {val_acc_after:.2f}%")
except NameError:
    print("Error: 'quantized_model' is not defined. Make sure to run the cell above that performs quantization first.")

In [ ]:
def print_size_of_model_params(m):
    total_params = 0
    for param in m.parameters():
        total_params += param.numel() * param.element_size()
    return total_params / (1024 * 1024) # Convert to MB

print(f"\nModel Size BEFORE (param memory): {print_size_of_model_params(model):.2f} MB")
print(f"Model Size AFTER (param memory): {print_size_of_model_params(quantized_model):.2f} MB")

In [ ]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

def get_predictions(loader, eval_model):
    eval_model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            B, P, C, H, W = inputs.shape
            inputs_merged = inputs.view(B * P, C, H, W)

            outputs = eval_model(inputs_merged)
            logits = outputs.logits
            logits = logits.view(B, P, -1)
            logits = logits.mean(dim=1)

            _, predicted = torch.max(logits, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

print("Evaluating Test Set Metrics (Original Model)...")
y_true, y_pred = get_predictions(test_loader, model)

# Calculate Macro Averages
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

# Calculate Specificity from Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
num_classes = cm.shape[0]
specificities = []
for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fp + fn)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificities.append(specificity)

macro_specificity = np.mean(specificities)

print(f"Precision (Macro):   {precision:.4f}")
print(f"Recall (Macro):      {recall:.4f}")
print(f"Specificity (Macro): {macro_specificity:.4f}")
print(f"F1 Score (Macro):    {f1:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Plot the confusion matrix using the true and predicted labels
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    cmap=plt.cm.Blues,
    ax=ax
)
plt.title("Confusion Matrix")
plt.show()


### Evaluation Metrics for Quantized Model
Now let's compute the detailed metrics and confusion matrix for the `quantized_model`.

In [ ]:
print("Evaluating Test Set Metrics (Quantized Model)...")
y_true_quantized, y_pred_quantized = get_predictions(test_loader, quantized_model)

# Define class names for plotting
class_names = train_dataset.classes

# Calculate Macro Averages for quantized model
precision_q = precision_score(y_true_quantized, y_pred_quantized, average='macro')
recall_q = recall_score(y_true_quantized, y_pred_quantized, average='macro')
f1_q = f1_score(y_true_quantized, y_pred_quantized, average='macro')

# Calculate Specificity from Confusion Matrix for quantized model
cm_q = confusion_matrix(y_true_quantized, y_pred_quantized)
num_classes_q = cm_q.shape[0]
specificities_q = []
for i in range(num_classes_q):
    tp_q = cm_q[i, i]
    fn_q = np.sum(cm_q[i, :]) - tp_q
    fp_q = np.sum(cm_q[:, i]) - tp_q
    tn_q = np.sum(cm_q) - (tp_q + fp_q + fn_q)
    specificity_q = tn_q / (tn_q + fp_q) if (tn_q + fp_q) > 0 else 0.0
    specificities_q.append(specificity_q)

macro_specificity_q = np.mean(specificities_q)

print(f"Precision (Macro, Quantized):   {precision_q:.4f}")
print(f"Recall (Macro, Quantized):      {recall_q:.4f}")
print(f"Specificity (Macro, Quantized): {macro_specificity_q:.4f}")
print(f"F1 Score (Macro, Quantized):    {f1_q:.4f}")
print("\nDetailed Classification Report (Quantized Model):")
print(classification_report(y_true_quantized, y_pred_quantized, target_names=class_names))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Plot the confusion matrix for the quantized model
fig_q, ax_q = plt.subplots(figsize=(8, 6))
disp_q = ConfusionMatrixDisplay.from_predictions(
    y_true_quantized,
    y_pred_quantized,
    cmap=plt.cm.Blues,
    ax=ax_q,
    display_labels=class_names
)
plt.title("Confusion Matrix -  Swin Tiny")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data for the models
models = ['MobileNet', 'Swin Tiny', 'Fusion (ELIX)']
accuracies = [96.7, 97.8, 98.08]

# Creating the bar chart
plt.figure(figsize=(10, 6))
# New distinct colors
colors = ['#4A90E2', '#50E3C2', '#B8E986']
bars = plt.bar(models, accuracies, color=colors)

# Formatting the chart
plt.ylim(90, 100)
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison')

# Adding labels on top of the bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval}%', ha='center', va='bottom', fontweight='bold')

# Removed the grid per request
plt.show()